Processing Categories to Apparatus as well as incidents

In [1]:
import geopandas as gpd
import pandas as pd
incidents_df= gpd.read_file("data/Vanderbilt_Fire/FireIncidents.geojson")
zones=gpd.read_file("data/beats_shpfile.geojson")
apparatus_df = pd.read_csv("data/Vanderbilt_Fire/Apparatus.csv")

NFDResponse = pd.read_csv("data/NFDResponse.csv")
cat_to_app = pd.read_csv("data/cat_to_apparatus.csv")



/var/folders/tf/7k8vj9w17715wv9yqsqn7pm00000gq/T/ipykernel_78216/1833893164.py:5: DtypeWarning: Columns (5,6,7,8,9,10,11,13,14,15,17,49,161,166) have mixed types. Specify dtype option on import or set low_memory=False.
  apparatus_df = pd.read_csv("data/Vanderbilt_Fire/Apparatus.csv")


In [2]:
apparatus_df.Incident_ID_Internal = apparatus_df.Incident_ID_Internal.astype(int)
apparatus_df=apparatus_df[apparatus_df.Incident_ID_Internal.isin(incidents_df['IncidentIDInternal'].astype(int))]
app_to_stations=pd.read_csv('data/ApparatusID_to_Station.csv')
app_to_stations.rename(columns={'Station': 'Facility Name'}, inplace=True)
app_to_stations.loc[app_to_stations['Facility Name'].str.contains('Goodlettsville Fire', na=False), 'Facility Name'] = 41
app_to_stations = app_to_stations[pd.to_numeric(app_to_stations['Facility Name'], errors='coerce').notna()]
app_to_stations['Facility Name'] = app_to_stations['Facility Name'].apply(lambda x: f"Station {int(x):02d}")


In [3]:
#split rows with "/" in Apparatus_Resource_ID
app_to_stations['Apparatus_Resource_ID'] = app_to_stations['Apparatus_Resource_ID'].str.split('/')
app_to_stations = app_to_stations.explode('Apparatus_Resource_ID')


In [4]:
apparatus=pd.merge(apparatus_df, app_to_stations, on='Apparatus_Resource_ID', how='left')
apparatus.sort_values(['Incident_ID_Internal','Apparatus_Resource_Arrival_Sequence_Number_By_Apparatus_Type'], inplace=True)
apparatus=apparatus[apparatus['Facility Name'].notna()].groupby('Incident_ID_Internal').first().reset_index()

In [5]:
incidents_df['IncidentIDInternal']=incidents_df['IncidentIDInternal'].astype(int)
incidents_df=pd.merge(incidents_df,apparatus, left_on='IncidentIDInternal',right_on='Incident_ID_Internal')
# incidents_df[incidents_df['IncidentIDInternal'].astype(int).isin(apparatus['Incident_ID_Internal'])]


In [6]:

type_cats=['EMSApparatusCount', 'OtherApparatusCount',
       'SuppressionApparatusCount', 'OtherPersonnelCount','PrimaryActionTaken']
incidents_export=pd.merge(incidents_df,pd.merge(cat_to_app,NFDResponse, left_on= "MappedCategory", right_on="Category", how="inner"), on='IncidentType', how='left')

In [7]:
incidents_export['response_time'] = incidents_export['AlarmLastUnitClearTime'] - incidents_export['AlarmFirstUnitArriveTime']
incidents_export['resolution_time'] = (incidents_export['LastUnitClearedDate'] - incidents_export['PSAPDate']).dt.total_seconds()
incidents_export = incidents_export[( incidents_export['SuppressionPersonnelCount'].notna())&(incidents_export['EMSApparatusCount'].notna())& ((incidents_export['response_time'].notna())&(incidents_export['response_time']> 0)) &(incidents_export['AlarmFirstUnitArriveTime']>0) & (incidents_export['AlarmLastUnitClearTime']>0)]
grouped_incidents = incidents_export.groupby('Enum')


for name, group in grouped_incidents:
    q99 = group['response_time'].quantile(0.99)
    q98= group['AlarmFirstUnitArriveTime'].quantile(0.99)
    incidents_export = incidents_export[~((incidents_export['Enum'] == name) & (incidents_export['response_time'] > q99))]
    incidents_export = incidents_export[~((incidents_export['Enum'] == name) & (incidents_export['AlarmFirstUnitArriveTime'] > q98))]

In [8]:
metrics = ["response_time"]
metrics_desc = []

for metric in metrics:
    incidents_export[metric] = incidents_export[metric].astype(float)  # Ensure numeric type
    summary = (
        incidents_export
          .groupby("Enum")[metric]
          .describe()
    )
    summary.Name = metric
    summary = summary.reset_index()
    summary.to_csv(f"data/exploratory_analysis/{metric}_summary.csv", index=False)
    # print(f"Summary for {metric} saved to data/exploratory_analysis/{metric}_summary.csv")incidents_export

In [9]:
action_severity_map = {
    "Investigate": "Low",
    "Provide manpower": "Low",
    "Extinguishment by fire service personnel": "High",
    "Search & rescue, other": "High",
    "Rescue, remove from harm": "High",
    "Provide basic life support (BLS)": "Moderate",
    "Emergency medical services, other": "Moderate",
    "Provide first aid & check for injuries": "Moderate",
    "Assistance, other": "Low",
    "Standby  (Staged on Scene)": "Low",
    "Assist physically disabled": "Moderate",
    "Hazardous materials spill control and confinement": "High",
    "Provide advanced life support (ALS)": "High",
    "Investigate fire out on arrival": "Low",
    "HazMat detection, monitoring, sampling, & analysis": "High",
    "Forcible entry": "Moderate",
    "Extricate, disentangle": "High",
    "Information, investigation & enforcement, other": "Low",
    "Shut down system": "Low",
    "Incident command": "Critical",
    "Assist animal": "Low",
    "Control traffic": "Low",
    "Secure property": "Low",
    "Ventilate B (Horizontal Ventilation)": "Moderate",
    "Notify other agencies.": "Low",
    "Remove hazard": "Moderate",
    "Refer to proper authority": "Low",
    "Provide information to public or media": "Low",
    "Restore fire alarm system": "Low",
    "Search": "Moderate",
    "Establish safe area": "Moderate",
    "Ventilate C (Smoke Removal Only)": "Moderate",
    "Action taken, other": "Low",
    "Provide apparatus": "Moderate",
    "Evacuate area": "High",
    "Remove water": "Low",
    "Identify, analyze hazardous materials": "High",
    "Provide equipment": "Moderate",
    "Hazardous materials leak control & containment": "High",
    "Enforce codes": "Low",
    "Restore sprinkler or fire protection system": "Low",
    "Fires, rescues & hazardous conditions, other": "High",
    "Remove hazardous materials": "High",
    "Transport person": "Moderate",
    "Systems and services, other": "Low",
    "Decontaminate persons or equipment": "High",
    "Recover body": "High",
    "Determine if materials are non-hazardous": "Low",
    "Provide water": "Moderate",
    "Ventilate A (Vertical Ventilation)": "Moderate",
    "Control fire (wildland)": "Critical",
    "Salvage & overhaul": "Moderate",
    "Hazardous condition, other": "Moderate",
    "Assess severe weather or natural disaster damage": "High",
    "Fill-in or moveup  (Back up)": "Low",
    "Operate apparatus or vehicle": "Low",
    "Decontaminate occupancy or area": "High",
    "Provide air supply": "Low",
    "Restore municipal services": "Moderate",
    "Contain fire (wildland)": "Critical",
    "Manage prescibed fire (wildland)": "High",
    "Provide light or electrical power": "Low",
    "Cancelled en route": "Low",
    "Establish fire lines (wildfire)": "Critical",
    "Confine fire (wildland)": "Critical",
    "Control crowd": "Low",
    "Fire control or extinguishment, other": "High"
}


In [10]:
incidents_export['incident_level'] = incidents_export['PrimaryActionTaken'].map(action_severity_map)
low = (incidents_export['incident_level'].isnull()) & (incidents_export['EMSApparatusCount'] >= 0) & (incidents_export['EMSApparatusCount'] <= 2) & (incidents_export['SuppressionApparatusCount'] == 0) 
moderate = (incidents_export['incident_level'].isnull()) & ((incidents_export['EMSApparatusCount'] > 2) | ((incidents_export['SuppressionApparatusCount'] >= 0) & (incidents_export['SuppressionApparatusCount'] <= 2)))
high = (incidents_export['incident_level'].isnull()) & (incidents_export['SuppressionApparatusCount'] > 2) & (incidents_export['SuppressionApparatusCount']<= 5) 
critical = (incidents_export['incident_level'].isnull()) & (incidents_export['SuppressionApparatusCount'] > 5) 
incidents_export.loc[low, 'incident_level'] = 'Low'
incidents_export.loc[moderate, 'incident_level'] = 'Moderate'
incidents_export.loc[high, 'incident_level'] = 'High'
incidents_export.loc[critical, 'incident_level'] = 'Critical'


In [11]:

# rename_cats=['Latitude','Longitude','NFIRSType','incident','PSAPDate','Enum']
cats=['incident_id','lat','lon','incident_type','incident_level','datetime','category']
incidents_export.rename(columns={'Longitude': 'lon', 'Latitude': 'lat','IncidentType': 'incident_type','PSAPDate': 'datetime','Enum': 'category'}, inplace=True)
incidents_export = incidents_export.dropna(subset=['lat', 'lon', 'incident_type', 'incident_level', 'datetime','Facility Name'])
incidents_export.sort_values('datetime',inplace=True)
incidents_export.reset_index(drop=True, inplace=True)
incidents_export.reset_index(names='incident_id', inplace=True)

In [12]:
incidents_export['datetime']=incidents_export['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
incidents_export['category'] = incidents_export['category'].str.strip()
incidents_export['incident_type'] = incidents_export['incident_type'].replace({',':' - ' }, regex=True)



In [13]:
incidents_export[cats].to_csv("data/incidents_export_apparatus.csv", index=False)

In [ ]:

incident_resolution_df = incidents_export[['incident_id', 'AlarmArriveTime','response_time', 'resolution_time','Facility Name']].copy()
incident_resolution_df.to_csv("data/incident_resolution_times.csv", index=False)

In [ ]:
org_dest=incidents_export[['IncidentNumber','lat','lon','Facility Name']]
org_dest.rename(columns={'lat':'dest_lat','lon':'dest_lon'}, inplace=True)
stations=pd.read_csv("data/stations.csv")
org_dest=org_dest.merge(stations[['Facility Name','lat','lon']], on='Facility Name', how='left')
org_dest.rename(columns={'lat':'org_lat','lon':'org_lon'}, inplace=True)
org_dest=org_dest[['IncidentNumber','org_lat','org_lon','dest_lat','dest_lon',]]


/var/folders/tf/7k8vj9w17715wv9yqsqn7pm00000gq/T/ipykernel_58364/992794729.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  org_dest.rename(columns={'lat':'dest_lat','lon':'dest_lon'}, inplace=True)


In [ ]:
org_dest

,IncidentNumber,org_lat,org_lon,dest_lat,dest_lon
0,FFD220101000005,36.124721,-86.697547,36.135850,-86.724410
1,FFD220101000007,36.216178,-86.801437,36.216591,-86.790614
2,FFD220101000008,36.179181,-86.811177,36.167447,-86.797420
3,FFD220101000009,36.179181,-86.811177,36.179460,-86.798507
4,FFD220101000014,36.067387,-86.630693,36.045748,-86.674218
...,...,...,...,...,...
495830,FFD250722100602,36.186711,-86.768415,36.164969,-86.769906
495831,FFD250722100603,36.186711,-86.768415,36.205670,-86.765780
495832,FFD250722100604,36.068452,-86.716910,36.059820,-86.715906
495833,FFD250722100605,36.258980,-86.715789,36.260909,-86.712993


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def get_travel_time(origin_lon, origin_lat, dest_lon, dest_lat, osrm_url="http://localhost:8080"):
    """Get travel time in seconds using OSRM container"""
    try:
        url = f"{osrm_url}/route/v1/driving/{origin_lon},{origin_lat};{dest_lon},{dest_lat}"
        response = requests.get(url, params={'overview': 'false'}, timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            if data['code'] == 'Ok':
                return data['routes'][0]['duration']
        return None
    except:
        return None

def add_travel_times_fast(df, origin_lon_col, origin_lat_col, dest_lon_col, dest_lat_col, max_workers=20):
    """Add travel time column to dataframe using parallel processing"""
    def get_time_for_row(args):
        idx, row = args
        time.sleep(0.01)  # Small delay to avoid overwhelming server
        travel_time = get_travel_time(row[origin_lon_col], row[origin_lat_col], 
                                    row[dest_lon_col], row[dest_lat_col])
        return idx, travel_time
    
    # Create list of (index, row) tuples
    row_data = [(idx, row) for idx, row in df.iterrows()]
    
    # Process in parallel
    travel_times = [None] * len(df)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = [executor.submit(get_time_for_row, row_item) for row_item in row_data]
        
        # Collect results with progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating travel times"):
            try:
                idx, travel_time = future.result()
                travel_times[idx] = travel_time
            except Exception as e:
                print(f"Error processing row: {e}")
    
    # Add results to dataframe
    df_copy = df.copy()
    df_copy['travel_time_seconds'] = travel_times
    df_copy['travel_time_minutes'] = [t/60 if t is not None else None for t in travel_times]
    
    # Print stats
    successful = sum(1 for t in travel_times if t is not None)
    print(f"Success rate: {successful}/{len(travel_times)} ({successful/len(travel_times)*100:.1f}%)")
    
    return df_copy

# df_with_times = add_travel_times_fast(org_dest, 'org_lon', 'org_lat', 'dest_lon', 'dest_lat')

# df_with_times = df_with_times.merge(incidents_export[['IncidentNumber','datetime','AlarmArriveTime','Facility Name']], left_on='IncidentNumber', right_on='IncidentNumber', how='left')
# df_with_times.rename(columns={'travel_time_seconds': 'osrm_travel_time_seconds', 'travel_time_minutes': 'osrm_travel_time_minutes'}, inplace=True)
# df_with_times.to_csv("data/incidents_with_travel_times_OSRM.csv", index=False)

Calculating travel times: 100%|██████████| 495835/495835 [09:51<00:00, 838.05it/s]  


Success rate: 487876/495835 (98.4%)


In [ ]:
import numpy as np
#get the metrics summary of bias, variance, rmse, mae, mape
def metrics_summary(df: pd.DataFrame,
                    actual_col: str = "Alarm_Time",
                    pred_col: str = "travel_time_seconds") -> pd.Series:
    """
    Returns bias, variance (of error), RMSE, MAE, MAPE as a pandas Series.
    - bias = mean(pred - actual)
    - variance = var(pred - actual), ddof=0
    - RMSE = sqrt(mean((pred - actual)^2))
    - MAE = mean(|pred - actual|)
    - MAPE = mean(|(pred - actual)/actual|)*100, computed only where actual>0
    """
    y_true = df[actual_col]
    y_pred = df[pred_col]

    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask].to_numpy(dtype=float)
    y_pred = y_pred[mask].to_numpy(dtype=float)

    err = y_pred - y_true

    bias = np.nanmean(err)
    var_err = np.nanvar(err)  # population variance
    rmse = np.sqrt(np.nanmean(err**2))
    mae = np.nanmean(np.abs(err))

    # MAPE on positive actuals to avoid div-by-zero
    pos = y_true > 0
    mape = np.nan if not np.any(pos) else np.nanmean(np.abs(err[pos] / y_true[pos])) * 100.0

    return pd.Series({
        "bias_seconds": bias,
        "variance_seconds2": var_err,
        "rmse_seconds": rmse,
        "mae_seconds": mae,
        "mape_percent": mape
    })

# summary = metrics_summary(df_with_times, actual_col="AlarmArriveTime", pred_col="travel_time_seconds")


In [ ]:
features=['incident_id',
 'incident_type',
 'datetime',
 'lat',
 'lon',
'category',
 'response_time',
 'resolution_time',
 'Facility Name',
 'AlarmArriveTime',
 'NFIRSType'
 ]
apparatus_features=['Engine_ID', 'Truck', 'Rescue', 'Hazard', 'Squad', 'FAST', 'Medic',
       'Brush', 'Boat', 'UTV', 'REACH', 'Chief']
incidents=gpd.GeoDataFrame(incidents_export, geometry=gpd.points_from_xy(incidents_export.lon, incidents_export.lat), crs="EPSG:4326")
incidents=incidents[features + ['geometry']+apparatus_features]
incidents=gpd.sjoin(incidents, zones, how="left", predicate='within').drop(columns=['index_right'])

incidents.dropna(subset=['ZONE_ID'], inplace=True)

incidents.drop(columns=['geometry','ZONE','TYPE','NAME','Facility Name','AlarmArriveTime','NFIRSType']+apparatus_features).to_csv("data/incidents_data_for_modeling.csv", index=False)

In [ ]:
stations= gpd.read_file("data/stations.csv")
stations=gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.lon, stations.lat), crs="EPSG:4326")
stations=stations[['Facility Name', 'lat', 'lon', 'geometry']]
stations=stations.sjoin( zones[['ZONE_ID','geometry']], how="left", predicate='within').drop(columns=['index_right'])
incidents=incidents.merge(stations[['Facility Name', 'lat', 'lon','ZONE_ID']], left_on='Facility Name', right_on='Facility Name', how='left',suffixes=('', '_station'))

In [ ]:

travel_time_features=['incident_id','incident_type','category','NFIRSType','datetime','lat','lon','ZONE_ID','Facility Name','lat_station','lon_station','ZONE_ID_station','AlarmArriveTime'] +apparatus_features
incidents[travel_time_features].to_csv("data/incidents_travel_time_features.csv", index=False)